# Experiments

Each experiment trains a model from scratch with `run_experiment`. Each run is saved in `models/<name>/`.
Training uses the GPU when one is available.

Every run reports Hits@20 on the training, validation and test splits after the last epoch.
No setting is chosen using the test split.

Run `pip install -e .` once from the project root before using this notebook.

In [ ]:
from functools import partial
from pathlib import Path

import itertools

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

from ddi_gnn.data import load_ddi
from ddi_gnn.plotting import plot_learning
from ddi_gnn.train import run_experiment
from ddi_gnn.utils import get_device

DATA_ROOT = "../datasets"
SAVE_DIR = "../models"

print(f"Training on: {get_device()}")

# Every run in this notebook saves to SAVE_DIR.
run = partial(run_experiment, save_dir=SAVE_DIR)

In [ ]:
def summarize(metrics: pd.DataFrame) -> dict:
    """Returns the Hits@20 of each split after the last epoch."""
    hits = metrics[metrics["metric"] == "hits@20"].sort_values("epoch")
    return hits.groupby("split")["mean"].last().to_dict()

## Dataset

A random subset of 500 drugs keeps these experiments fast.

In [ ]:
small_data = load_ddi(
    root=DATA_ROOT,
    node_limit=500,
    generator=torch.Generator().manual_seed(42),
)
print(f"{small_data.n_nodes} drugs, {len(small_data.splits['train'].pos)} training interactions")

## Baseline model

The simplest setting: binary log loss, no self-edge in the last layer, no degree normalization.

In [ ]:
BASE_MODEL = {
    "embedding_dim": 128,
    "dropout_rate": 0.3,
    "last_layer_self": False,
    "degree_norm": False,
}

_, metrics = run(
    "baseline_bce", small_data, BASE_MODEL, loss="bce", num_epochs=500, eval_every=1
)
plot_learning(metrics)
plt.show()
summarize(metrics)

## Changing to the AUC loss

In [ ]:
_, metrics = run(
    "baseline_auc", small_data, BASE_MODEL, loss="auc", num_epochs=500, eval_every=1
)
plot_learning(metrics)
plt.show()
summarize(metrics)

## Varying the embedding dimension

In [ ]:
rows = []
for embedding_dim in [64, 128, 256, 512]:
    _, metrics = run(
        f"sweep_embedding_dim_{embedding_dim}",
        small_data,
        {**BASE_MODEL, "embedding_dim": embedding_dim},
        loss="auc",
    )
    rows.append({"embedding_dim": embedding_dim, **summarize(metrics)})

embedding_results = pd.DataFrame(rows)
embedding_results

In [ ]:
data = embedding_results.melt(
    id_vars="embedding_dim", var_name="split", value_name="hits@20"
)

plt.figure(figsize=(7, 3.5))
sns.barplot(data=data, x="embedding_dim", y="hits@20", hue="split", palette="colorblind")
plt.ylim(0, 1)
plt.xlabel("Embedding Dimensions")
plt.ylabel("Hits@20 (last epoch)")
plt.show()

## Varying several hyperparameters

The grid has 3 x 2 x 2 x 3 = 36 combinations.

In [ ]:
sweep_grid = {
    "dropout_rate": [0, 0.3, 0.5],
    "last_layer_self": [True, False],
    "degree_norm": [True, False],
    "n_mlp_layers": [1, 2, 3],
}
combinations = [
    dict(zip(sweep_grid, values)) for values in itertools.product(*sweep_grid.values())
]
pd.DataFrame(combinations).head()

In [ ]:
rows = []
for params in combinations:
    name = "sweep_all_" + "_".join(f"{key}-{value}" for key, value in params.items())
    _, metrics = run(
        name,
        small_data,
        {**BASE_MODEL, "embedding_dim": 256, **params},
        loss="auc",
    )
    rows.append({**params, **summarize(metrics), "run": name})

sweep_results = pd.DataFrame(rows)
sweep_results.sort_values("valid", ascending=False).head(10)

In [ ]:
def conv_layer_label(row) -> str:
    if row["last_layer_self"] and row["degree_norm"]:
        return "self-edge, norm"
    if row["last_layer_self"]:
        return "self-edge, no norm"
    if row["degree_norm"]:
        return "no self-edge, norm"
    return "no self-edge, no norm"


data = sweep_results.melt(
    id_vars=list(sweep_grid),
    value_vars=["train", "valid", "test"],
    var_name="split",
    value_name="hits@20",
)
data["conv_layer"] = data.apply(conv_layer_label, axis=1)

grid = sns.relplot(
    data=data,
    x="conv_layer",
    y="hits@20",
    row="dropout_rate",
    col="n_mlp_layers",
    hue="split",
    palette="colorblind",
    facet_kws={"margin_titles": True},
    height=2,
)
grid.figure.subplots_adjust(wspace=0.1, hspace=0)
for ax in grid.axes.flat:
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
grid.set_axis_labels("", "Hits@20 (last epoch)")
plt.show()

Learning curves of the configuration with the best validation Hits@20:

In [ ]:
best = sweep_results.sort_values("valid", ascending=False).iloc[0]
print(best.drop("run").to_dict())

plot_learning(pd.read_csv(Path(SAVE_DIR) / best["run"] / "metrics.csv"))
plt.show()

## Training on a larger dataset

A random subset of 2134 drugs (half of all drugs). Set the model settings below from the sweep results above.
A larger batch size keeps each epoch fast.

In [ ]:
large_data = load_ddi(
    root=DATA_ROOT,
    node_limit=2134,
    generator=torch.Generator().manual_seed(42),
)
print(f"{large_data.n_nodes} drugs, {len(large_data.splits['train'].pos)} training interactions")

In [ ]:
LARGE_MODEL = {
    "embedding_dim": 512,
    "dropout_rate": 0.3,
    "last_layer_self": True,
    "degree_norm": False,
    "n_mlp_layers": 2,
}

_, metrics = run(
    "larger_data",
    large_data,
    LARGE_MODEL,
    loss="auc",
    num_epochs=500,
    batch_size=16384,
    eval_every=25,
)
plot_learning(metrics)
plt.show()
summarize(metrics)